# Tensor and image columns in nested-pandas and LSDB

This notebook demos the new tensor-backed columns: numpy arrays as first-class
column values in `NestedFrame`s and LSDB catalogs, with full parquet round trips.

Requires the `tensor-image-columns` branches of **nested-pandas** and **lsdb**.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import nested_pandas as npd
from nested_pandas import (
    TensorDtype, TensorArray, TensorSeries,
    ImageDtype, ImageSeries,
)

rng = np.random.default_rng(42)

## Background: Arrow extension types

Arrow's type system is a closed set — there is no way to add a truly new physical
type. An **extension type** is Arrow's escape hatch: an annotation layered on top
of an ordinary *storage type*. It consists of exactly three things:

1. a **storage type** — the real physical layout (a list, a struct, ...) that holds the bytes,
2. a **name** — a namespaced string like `arrow.fixed_shape_tensor` or `nested_pandas.image`,
3. a **metadata blob** — arbitrary bytes the type serializes for itself.

The name and metadata travel with the schema (in files they become two key-value
entries on the field). A reader that has the name *registered* reconstructs the
extension type and any richer behavior on top; a reader that doesn't simply sees
the storage type — the data is never hostage to the annotation.

Arrow ships a few *canonical* extension types with reserved `arrow.*` names that
many implementations understand. The one we build on:

In [2]:
import pyarrow as pa

canonical = pa.fixed_shape_tensor(pa.float32(), (25, 25))
print("extension name:", canonical.extension_name)
print("storage type:  ", canonical.storage_type)

extension name: arrow.fixed_shape_tensor
storage type:   fixed_size_list<item: float>[625]


### Defining our own

A custom extension type subclasses `pa.ExtensionType`, picks the storage type and
name, and defines its metadata through a serialize/deserialize pair. Ours keep a
small JSON blob with the number of dimensions and (for fixed-shape columns) the
declared shape — `__arrow_ext_serialize__` writes it, `__arrow_ext_deserialize__`
parses it back when a file is read:

In [3]:
import inspect
from nested_pandas.tensor.arrow_ext import TensorType, ImageType

print(inspect.getsource(TensorType.__arrow_ext_serialize__))
print(inspect.getsource(TensorType.__arrow_ext_deserialize__))

    def __arrow_ext_serialize__(self) -> bytes:
        metadata: dict = {
            "ndim": self._ndim,
            "shape": list(self._shape) if self._shape is not None else None,
        }
        return json.dumps(metadata).encode()

    @classmethod
    def __arrow_ext_deserialize__(cls, storage_type, serialized):
        metadata = json.loads(serialized.decode())
        value_type = storage_type.field("data").type.value_type
        shape = metadata.get("shape")
        return cls(
            value_type,
            metadata["ndim"],
            shape=tuple(shape) if shape is not None else None,
        )



In [4]:
img_type = ImageType(pa.float32(), 2, shape=(25, 25))
print("extension name:", img_type.extension_name)   # the subclass overrides only the name
print("storage type:  ", img_type.storage_type)
print("metadata:      ", img_type.__arrow_ext_serialize__().decode())

extension name: nested_pandas.image
storage type:   struct<data: list<item: float>, shape: list<item: int32>>
metadata:       {"ndim": 2, "shape": [25, 25]}


An extension *array* is just a storage array with the type attached — building one
never copies the data:

In [5]:
ragged_type = TensorType(pa.float32(), 2)
storage = pa.array(
    [{"data": [1.0, 2.0, 3.0, 4.0], "shape": [2, 2]}, None],
    type=ragged_type.storage_type,
)
extension_array = pa.ExtensionArray.from_storage(ragged_type, storage)
extension_array.type

TensorType(StructType(struct<data: list<item: float>, shape: list<item: int32>>))

Both of our types are registered with pyarrow when nested-pandas is imported
(`pa.register_extension_type`), which is what lets parquet reads reconstruct them
by name. We'll see exactly how the name and metadata are stored in a file — and
what an *unregistered* reader sees — in the inspection section further down.

## Fixed-shape tensor columns

A tensor column stores one `np.ndarray` per row. When every row shares one shape,
the dtype carries it (`tensor[float, (4, 4)]`) and the data is stored as the Arrow
canonical `arrow.fixed_shape_tensor` layout: one flat contiguous buffer.

In [6]:
stack = rng.normal(size=(6, 4, 4)).astype(np.float32)
tensors = TensorArray.from_stack(stack)
tensors.dtype

tensor[float, (4, 4)]

In [7]:
# Scalar access returns a zero-copy numpy view over the Arrow buffers
tensors[0]

array([[ 0.3047171 , -1.0399841 ,  0.7504512 ,  0.9405647 ],
       [-1.9510351 , -1.3021795 ,  0.1278404 , -0.3162426 ],
       [-0.01680116, -0.8530439 ,  0.879398  ,  0.7777919 ],
       [ 0.0660307 ,  1.1272413 ,  0.46750933, -0.85929245]],
      dtype=float32)

In [8]:
# ...and so does restacking the whole column into one (n, 4, 4) block
restacked = tensors.to_stack()
buffer_address = tensors._storage.chunk(0).buffers()[-1].address
print("zero-copy:", restacked.__array_interface__["data"][0] == buffer_address)
restacked.shape

zero-copy: True


(6, 4, 4)

## Variable-shape (ragged) tensors

With `ndim` instead of `shape`, each row can have its own shape. Missing values
are `pd.NA`, at the row level.

In [9]:
ragged = pd.array(
    [rng.normal(size=(3, 5)), rng.normal(size=(7, 2)), None],
    dtype=TensorDtype("float64", ndim=2),
)
ragged

<TensorArray>
[[3×5] float64, [7×2] float64, <NA>]
Length: 3, dtype: tensor[double, ndim=2]

In [10]:
print(ragged.shapes)
ragged.isna()

[[3 5]
 [7 2]
 [0 0]]


array([False, False,  True])

## Tensor columns in a NestedFrame

Tensor columns live alongside regular and nested columns. Column access returns
a `TensorSeries` with tensor-aware accessors.

In [11]:
n_obj = 6
nf = npd.NestedFrame({
    "id": np.arange(n_obj),
    "mag": rng.uniform(18, 22, n_obj),
})

# a nested light-curve column, for good measure
lightcurves = npd.NestedFrame({
    "index": np.repeat(np.arange(n_obj), 10),
    "mjd": np.tile(np.linspace(0, 30, 10), n_obj),
    "flux": rng.normal(1.0, 0.1, n_obj * 10),
}).set_index("index")
nf = nf.join_nested(lightcurves, "lightcurve")

nf["features"] = tensors
nf

id        mag                                 lightcurve       features
0   0  18.578097  [{mjd: 0.0, flux: 0.862331}; …] (10 rows)  [4×4] float32
1   1  18.413612  [{mjd: 0.0, flux: 0.901046}; …] (10 rows)  [4×4] float32
2   2  20.350578  [{mjd: 0.0, flux: 0.896435}; …] (10 rows)  [4×4] float32
3   3  18.682372  [{mjd: 0.0, flux: 0.927477}; …] (10 rows)  [4×4] float32
4   4  21.700480  [{mjd: 0.0, flux: 1.044607}; …] (10 rows)  [4×4] float32
5   5  20.324245  [{mjd: 0.0, flux: 1.021938}; …] (10 rows)  [4×4] float32

In [12]:
column = nf["features"]
print(type(column).__name__, column.tensor_shape, column.value_dtype)
column.to_stack().mean(axis=(1, 2))  # vectorized work on the whole column

TensorSeries (4, 4) float32


array([-0.05606465,  0.19604315,  0.09840017, -0.07285692, -0.0369159 ,
       -0.26618773], dtype=float32)

## Image columns

`ImageDtype`/`ImageSeries` are tensor columns with image semantics: same storage,
but cells render as PNG thumbnails in HTML reprs. Let's fake some stamps —
noisy Gaussian sources.

In [13]:
def make_stamp(rng, size=25):
    """A noisy Gaussian source with random position, width and brightness."""
    y, x = np.mgrid[0:size, 0:size]
    cy, cx = rng.uniform(size * 0.3, size * 0.7, 2)
    sigma = rng.uniform(1.5, 3.5)
    amplitude = rng.uniform(5, 50)
    stamp = amplitude * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * sigma**2))
    return (stamp + rng.normal(0, 0.5, stamp.shape)).astype(np.float32)

nf["stamp"] = pd.array(
    [make_stamp(rng) for _ in range(n_obj - 1)] + [None],  # one missing row
    dtype=ImageDtype("float32", shape=(25, 25)),
)
nf

id        mag                                 lightcurve       features  \
0   0  18.578097  [{mjd: 0.0, flux: 0.862331}; …] (10 rows)  [4×4] float32   
1   1  18.413612  [{mjd: 0.0, flux: 0.901046}; …] (10 rows)  [4×4] float32   
2   2  20.350578  [{mjd: 0.0, flux: 0.896435}; …] (10 rows)  [4×4] float32   
3   3  18.682372  [{mjd: 0.0, flux: 0.927477}; …] (10 rows)  [4×4] float32   
4   4  21.700480  [{mjd: 0.0, flux: 1.044607}; …] (10 rows)  [4×4] float32   
5   5  20.324245  [{mjd: 0.0, flux: 1.021938}; …] (10 rows)  [4×4] float32   

             stamp  
0  [25×25] float32  
1  [25×25] float32  
2  [25×25] float32  
3  [25×25] float32  
4  [25×25] float32  
5             <NA>

In [14]:
stamps = nf["stamp"]
stamps  # ImageSeries: thumbnails in notebooks, <NA> for the missing row

,stamp,
0,,[25×25] float32
1,,[25×25] float32
2,,[25×25] float32
3,,[25×25] float32
4,,[25×25] float32
5,<NA>,


In [15]:
# Missing rows fill with NaN when stacking into a pixel cube
cube = stamps.to_image_stack()
print(cube.shape, "| last row all-NaN:", np.isnan(cube[-1]).all())

(6, 25, 25) | last row all-NaN: True


## Serialization: parquet round trip

Plain fixed-shape tensor columns are written as the Arrow **canonical**
`arrow.fixed_shape_tensor` extension type, readable by any Arrow implementation.
Image columns are written as their own `nested_pandas.image` extension type
(ragged or nullable-fixed plain tensors as `nested_pandas.tensor`) — the type
name itself carries the identity, so image columns read back as image columns.

In [16]:
nf.to_parquet("tensor_demo.parquet")
schema = pq.read_schema("tensor_demo.parquet")
{name: str(schema.field(name).type) for name in ["features", "stamp"]}

{'features': 'extension<arrow.fixed_shape_tensor[value_type=float, shape=[4,4]]>',
 'stamp': 'extension<nested_pandas.image<ImageType>>'}

In [17]:
back = npd.read_parquet("tensor_demo.parquet")
back.dtypes

id                                   int64[pyarrow]
mag                                 double[pyarrow]
lightcurve    nested<mjd: [double], flux: [double]>
features                      tensor[float, (4, 4)]
stamp                        image[float, (25, 25)]
dtype: object

In [18]:
print(type(back["stamp"]).__name__)
print("features equal:", np.array_equal(back["features"].to_stack(), nf["features"].to_stack()))
back

ImageSeries
features equal: True


id        mag                                 lightcurve       features  \
0   0  18.578097  [{mjd: 0.0, flux: 0.862331}; …] (10 rows)  [4×4] float32   
1   1  18.413612  [{mjd: 0.0, flux: 0.901046}; …] (10 rows)  [4×4] float32   
2   2  20.350578  [{mjd: 0.0, flux: 0.896435}; …] (10 rows)  [4×4] float32   
3   3  18.682372  [{mjd: 0.0, flux: 0.927477}; …] (10 rows)  [4×4] float32   
4   4   21.70048  [{mjd: 0.0, flux: 1.044607}; …] (10 rows)  [4×4] float32   
5   5  20.324245  [{mjd: 0.0, flux: 1.021938}; …] (10 rows)  [4×4] float32   

             stamp  
0  [25×25] float32  
1  [25×25] float32  
2  [25×25] float32  
3  [25×25] float32  
4  [25×25] float32  
5             <NA>

A reader *without* nested-pandas degrades gracefully — it sees the plain storage
layouts (a fixed-size list of floats, or a `struct<data, shape>`), with the pixel
data fully intact:

In [19]:
{
    "features": str(schema.field("features").type.storage_type),
    "stamp": str(schema.field("stamp").type.storage_type),
}

{'features': 'fixed_size_list<item: float>[16]',
 'stamp': 'struct<data: list<item: float>, shape: list<item: int32>>'}

## Under the hood: inspecting the parquet file

Everything that makes these columns special lives in ordinary parquet metadata.
Let's poke at the file with plain `pyarrow.parquet` tooling, starting with the
*parquet-level* schema — the physical layout, which is all a non-Arrow reader sees.
Note that parquet has no fixed-size-list or tensor concept: the fixed-shape
`features` column is just a repeated float group, and `stamp` a group of two lists.

In [20]:
parquet_file = pq.ParquetFile("tensor_demo.parquet")
print(parquet_file.metadata)
print()
print(parquet_file.schema)

  created_by: parquet-cpp-arrow version 21.0.0
  num_columns: 7
  num_rows: 6
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 2635

required group field_id=-1 schema {
  optional int64 field_id=-1 id;
  optional double field_id=-1 mag;
  optional group field_id=-1 lightcurve {
    optional group field_id=-1 mjd (List) {
      repeated group field_id=-1 list {
        optional double field_id=-1 element;
      }
    }
    optional group field_id=-1 flux (List) {
      repeated group field_id=-1 list {
        optional double field_id=-1 element;
      }
    }
  }
  optional group field_id=-1 features (List) {
    repeated group field_id=-1 list {
      optional float field_id=-1 element;
    }
  }
  optional group field_id=-1 stamp {
    optional group field_id=-1 data (List) {
      repeated group field_id=-1 list {
        optional float field_id=-1 element;
      }
    }
    optional group field_id=-1 shape (List) {
      repeated group field_id=-1 list {
        optiona

The *Arrow-level* schema is reconstructed from the `ARROW:schema` entry the writer
embeds in the footer. There the extension types reappear, each with a name, a
storage type, and a small metadata blob — for our types, JSON carrying `ndim` and
the declared `shape` (which is how a nullable fixed-shape column knows its shape
on read):

In [21]:
schema = pq.read_schema("tensor_demo.parquet")
for name in ["features", "stamp"]:
    ext_type = schema.field(name).type
    print(f"{name}:")
    print("   type:         ", ext_type)
    print("   extension name:", ext_type.extension_name)
    print("   storage type:  ", ext_type.storage_type)
    if hasattr(ext_type, "__arrow_ext_serialize__"):
        print("   metadata:      ", ext_type.__arrow_ext_serialize__().decode())
    print()

features:
   type:          extension<arrow.fixed_shape_tensor[value_type=float, shape=[4,4]]>
   extension name: arrow.fixed_shape_tensor
   storage type:   fixed_size_list<item: float>[16]

stamp:
   type:          extension<nested_pandas.image<ImageType>>
   extension name: nested_pandas.image
   storage type:   struct<data: list<item: float>, shape: list<item: int32>>
   metadata:       {"ndim": 2, "shape": [25, 25]}



We can see exactly what a reader *without* nested-pandas installed sees by
unregistering the extension type and re-reading the schema: the column degrades to
its plain storage struct, and the extension identity survives as two
`ARROW:extension:*` key-value entries on the field, waiting for a reader that
understands them:

In [22]:
import pyarrow as pa

pa.unregister_extension_type("nested_pandas.image")
try:
    degraded = pq.read_schema("tensor_demo.parquet").field("stamp")
    print(degraded.type)
    for key, value in degraded.metadata.items():
        print(f"   {key.decode()} = {value.decode()}")
finally:
    from nested_pandas.tensor.arrow_ext import _register_extension_types
    _register_extension_types()  # put it back

struct<data: list<element: float>, shape: list<element: int32>>
   ARROW:extension:metadata = {"ndim": 2, "shape": [25, 25]}
   ARROW:extension:name = nested_pandas.image


Finally, the column-chunk statistics show that pixels are ordinary float leaf
columns — compressed pages with sizes and statistics, so column projection and
predicate pushdown on the neighboring columns work untouched. The stamp pixels
dominate the file, as they should:

In [23]:
row_group = parquet_file.metadata.row_group(0)
pd.DataFrame([
    {
        "column": (col := row_group.column(i)).path_in_schema,
        "physical_type": col.physical_type,
        "compression": col.compression,
        "compressed_bytes": col.total_compressed_size,
    }
    for i in range(row_group.num_columns)
])

,column,physical_type,compression,compressed_bytes
0,id,INT64,SNAPPY,122
1,mag,DOUBLE,SNAPPY,138
2,lightcurve.mjd.list.element,DOUBLE,SNAPPY,167
3,lightcurve.flux.list.element,DOUBLE,SNAPPY,638
4,features.list.element,FLOAT,SNAPPY,561
5,stamp.data.list.element,FLOAT,SNAPPY,17301
6,stamp.shape.list.element,INT32,SNAPPY,85


## Nested image columns: stamps inside a light curve

Nested column fields can themselves be image columns — e.g. one cutout per
*epoch*, packed into the light curve. `join_nested` handles the image column like
any other field, and the nested dtype records the image type of the sub-column:

In [24]:
n_epochs = 3
epochs = npd.NestedFrame({
    "index": np.repeat(np.arange(n_obj), n_epochs),
    "mjd": np.tile(np.linspace(0.0, 30.0, n_epochs), n_obj),
    "flux": rng.normal(1.0, 0.1, n_obj * n_epochs),
}).set_index("index")
epochs["stamp"] = pd.array(
    [make_stamp(rng, size=15) for _ in range(len(epochs))],
    dtype=ImageDtype("float32", shape=(15, 15)),
)

lc_nf = npd.NestedFrame({"id": np.arange(n_obj)}).join_nested(epochs, "lightcurve")
lc_nf["lightcurve"].dtype

nested<mjd: [double], flux: [double], stamp: [extension<nested_pandas.image<ImageType>>]>

In [25]:
lc_nf["lightcurve"].iloc[0]

,mjd,flux,stamp
0,0.0,0.960252,"{'data': [-1.3518775, -0.3106164, -0.2618453, ..."
1,15.0,1.005621,"{'data': [0.15453589, 1.0163597, -0.51106244, ..."
2,30.0,0.971863,"{'data': [1.5043628, 3.2545724, 3.6179945, 5.3..."


In [26]:
lc_nf

id                                         lightcurve
0   0  [{mjd: 0.0, flux: 0.960252, stamp: {'data': ar...
1   1  [{mjd: 0.0, flux: 0.975318, stamp: {'data': ar...
2   2  [{mjd: 0.0, flux: 0.837196, stamp: {'data': ar...
3   3  [{mjd: 0.0, flux: 1.168755, stamp: {'data': ar...
4   4  [{mjd: 0.0, flux: 0.976763, stamp: {'data': ar...
5   5  [{mjd: 0.0, flux: 1.05393, stamp: {'data': arr...

Flattened sub-column access (`"lightcurve.stamp"`) currently comes back as the raw
arrow-backed series — rehydrating it into a first-class image column is one line
(wiring this into the `.nest` accessor is future work):

In [45]:
import pyarrow as pa

print("flattened dtype:", lc_nf["lightcurve.stamp"].dtype)
epoch_stamps = ImageSeries(pd.Series(TensorArray(pa.array(lc_nf["lightcurve.stamp"])), name="stamp"))
epoch_stamps

flattened dtype: extension<nested_pandas.image<ImageType>>[pyarrow]


,stamp,
0,,[15×15] float32
1,,[15×15] float32
2,,[15×15] float32
3,,[15×15] float32
4,,[15×15] float32
5,,[15×15] float32
6,,[15×15] float32
7,,[15×15] float32
8,,[15×15] float32
9,,[15×15] float32


The whole structure — light curves with their per-epoch pixels — round-trips
through parquet with the image identity intact inside the nested type:

In [28]:
lc_nf.to_parquet("nested_stamps_demo.parquet")
back = npd.read_parquet("nested_stamps_demo.parquet")
print(back["lightcurve"].dtype)
original_cell = np.asarray(lc_nf["lightcurve.stamp"].iloc[0]["data"]).reshape(15, 15)
restored = TensorArray(pa.array(back["lightcurve.stamp"]))
print("pixels equal:", np.array_equal(restored[0], original_cell))

nested<mjd: [double], flux: [double], stamp: [extension<nested_pandas.image<ImageType>>]>
pixels equal: True


## Tensor and image columns in an LSDB catalog

The same columns work in distributed LSDB catalogs: `map_partitions` to attach
them, and `write_catalog`/`open_catalog` round-trips them through HATS parquet.

In [29]:
import lsdb

n_src = 100
source_df = pd.DataFrame({
    "id": np.arange(n_src),
    "ra": rng.uniform(0, 360, n_src),
    "dec": rng.uniform(-90, 90, n_src),
    "mag": rng.uniform(16, 24, n_src),
})
catalog = lsdb.from_dataframe(source_df, lowest_order=0, highest_order=1)

def add_stamps(partition):
    part_rng = np.random.default_rng(len(partition))
    stamps = [make_stamp(part_rng) for _ in range(len(partition))]
    partition["stamp"] = pd.array(stamps, dtype=ImageDtype("float32", shape=(25, 25)))
    return partition

meta = catalog.meta.copy()
meta["stamp"] = pd.array([], dtype=ImageDtype("float32", shape=(25, 25)))
catalog = catalog.map_partitions(add_stamps, meta=meta)
catalog.dtypes

id               int64[pyarrow]
ra              double[pyarrow]
dec             double[pyarrow]
mag             double[pyarrow]
stamp    image[float, (25, 25)]
dtype: object

In [30]:
catalog.compute().head(4)

Computing Catalog:   0%|          | 0/24 [00:00<?, ?it/s]

,id,ra,dec,mag,stamp
34398531294117442,5,52.326110,24.268970,21.970711,
183793300033391828,31,11.418175,38.318374,20.283953,
135362719653019486,47,65.142942,50.586013,20.950479,
74270502984238595,61,70.603328,26.678887,21.212953,


### Write and reopen as a HATS catalog

In [31]:
import shutil
shutil.rmtree("tensor_demo_catalog", ignore_errors=True)
catalog.write_catalog("tensor_demo_catalog", catalog_name="tensor_demo_catalog")

Writing Catalog:   0%|          | 0/36 [00:00<?, ?it/s]

In [32]:
reopened = lsdb.open_catalog("tensor_demo_catalog")
reopened.dtypes  # dtype known from the schema alone, before any compute

id               int64[pyarrow]
ra              double[pyarrow]
dec             double[pyarrow]
mag             double[pyarrow]
stamp    image[float, (25, 25)]
dtype: object

In [33]:
result = reopened.compute()
print(type(result["stamp"]).__name__, result["stamp"].dtype)
result.head(4)

Computing Catalog:   0%|          | 0/12 [00:00<?, ?it/s]

ImageSeries image[float, (25, 25)]


,id,ra,dec,mag,stamp
34398531294117442,5,52.326110,24.268970,21.970711,
183793300033391828,31,11.418175,38.318374,20.283953,
135362719653019486,47,65.142942,50.586013,20.950479,
74270502984238595,61,70.603328,26.678887,21.212953,


## Recap

- `TensorDtype`/`TensorArray` store one ndarray per row, fixed-shape (`shape=`) or
  ragged (`ndim=`), with zero-copy scalar access and `to_stack()`.
- `ImageDtype`/`ImageSeries` add image semantics and thumbnail reprs on the same
  storage.
- Both coexist with base and nested columns in `NestedFrame`, and survive parquet
  and full LSDB catalog round trips — canonical Arrow tensors where possible, the
  `nested_pandas.tensor` extension type (with its `kind` tag) everywhere else.